In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:

# EXPERIMENT 01 — Environment & Release Verification
# Goal: Verify the target Transformers release and available CPU/RAM.

import sys
import platform
import subprocess
import importlib.metadata as metadata
import os

print("=" * 65)
print("EXPERIMENT 01: TRANSFORMERS EXPORT INVESTIGATION")
print("=" * 65)

# 1. Python environment
print("\n[1] PYTHON ENVIRONMENT")
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

# 2. Hardware resources
print("\n[2] HARDWARE")
print("CPU cores:", os.cpu_count())

try:
    with open("/proc/meminfo") as f:
        mem = next(line for line in f if line.startswith("MemTotal"))
    ram_gb = int(mem.split()[1]) / 1024**2
    print(f"Total RAM: {ram_gb:.2f} GB")
except Exception as e:
    print("RAM detection:", e)

# 3. GPU status (informational only)
print("\n[3] GPU STATUS")
try:
    result = subprocess.run(
        ["nvidia-smi"],
        capture_output=True,
        text=True
    )
    print(result.stdout[:1500] if result.returncode == 0
          else "No NVIDIA GPU available. CPU experiment is sufficient.")
except FileNotFoundError:
    print("No NVIDIA GPU detected. Using CPU.")

# 4. Installed package versions
print("\n[4] INSTALLED PACKAGES")

for package in ["torch", "transformers", "onnx", "onnxruntime",
                "optimum", "numpy", "psutil"]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

# 5. Verify target release availability
print("\n[5] TARGET RELEASE VERIFICATION")

try:
    import urllib.request
    import json

    url = "https://pypi.org/pypi/transformers/json"
    with urllib.request.urlopen(url, timeout=15) as response:
        pypi_data = json.load(response)

    versions = pypi_data["releases"]
    target = "5.23.0"

    print("Latest published version:", pypi_data["info"]["version"])
    print(f"Target version {target} exists:", target in versions)

    if target in versions:
        print("Target release date:",
              versions[target][0].get("upload_time", "Unknown")
              if versions[target] else "No files listed")
    else:
        print("Available 5.x versions:",
              sorted([v for v in versions if v.startswith("5.")])[-10:])

except Exception as e:
    print("Release verification failed:", repr(e))

print("\n" + "=" * 65)
print("ENVIRONMENT CHECK COMPLETE")
print("=" * 65)


EXPERIMENT 01: TRANSFORMERS EXPORT INVESTIGATION

[1] PYTHON ENVIRONMENT
Python: 3.13.18
Platform: Linux-6.18.55+-x86_64-with-glibc2.39

[2] HARDWARE
CPU cores: 4
Total RAM: 31.35 GB

[3] GPU STATUS
No NVIDIA GPU detected. Using CPU.

[4] INSTALLED PACKAGES
torch: 2.11.0+cpu
transformers: 5.20.2
onnx: 1.23.2
onnxruntime: NOT INSTALLED
optimum: NOT INSTALLED
numpy: 2.1.3
psutil: 5.9.5

[5] TARGET RELEASE VERIFICATION
Latest published version: 5.23.0
Target version 5.23.0 exists: True
Target release date: 2026-11-13T14:10:36

ENVIRONMENT CHECK COMPLETE


In [3]:

# EXPERIMENT 01 — PyTorch Baseline
# Load the target Transformers release and generate reference embeddings.

%pip install -q transformers==5.23.0

import torch
import numpy as np
import time
import json
from pathlib import Path
from transformers import AutoTokenizer, AutoModel

# Reproducibility
torch.manual_seed(42)
torch.set_num_threads(4)

MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
OUTPUT_DIR = Path("/kaggle/working/experiment01")
OUTPUT_DIR.mkdir(exist_ok=True)

# Fixed test inputs for both implementations
texts = [
    "Artificial intelligence is transforming software development.",
    "Machine learning models require efficient inference for production.",
    "Hugging Face provides tools for loading and deploying pretrained models.",
    "ONNX Runtime can execute models exported from PyTorch."
]

print("Transformers version:", __import__("transformers").__version__)
print("Loading model:", MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID)
model.eval()

# Tokenization
inputs = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# PyTorch inference
with torch.inference_mode():
    start = time.perf_counter()

    outputs = model(**inputs)

    # Attention-mask-aware mean pooling
    token_embeddings = outputs.last_hidden_state
    attention_mask = inputs["attention_mask"].unsqueeze(-1)

    embeddings = (
        (token_embeddings * attention_mask).sum(dim=1)
        / attention_mask.sum(dim=1).clamp(min=1)
    )

    baseline_embeddings = embeddings.cpu().numpy()
    elapsed = time.perf_counter() - start

# Save reference outputs
np.save(OUTPUT_DIR / "pytorch_embeddings.npy", baseline_embeddings)

with open(OUTPUT_DIR / "baseline_metadata.json", "w") as f:
    json.dump({
        "model": MODEL_ID,
        "transformers_version": __import__("transformers").__version__,
        "framework": "PyTorch",
        "device": "CPU",
        "input_count": len(texts),
        "embedding_shape": list(baseline_embeddings.shape),
        "inference_time_seconds": elapsed,
        "pooling": "attention_mask_mean"
    }, f, indent=2)

print("\n========== BASELINE RESULTS ==========")
print("Model:", MODEL_ID)
print("Embedding shape:", baseline_embeddings.shape)
print(f"Inference time: {elapsed * 1000:.2f} ms")
print("Saved:", OUTPUT_DIR / "pytorch_embeddings.npy")
print("Saved:", OUTPUT_DIR / "baseline_metadata.json")
print("First embedding (first 8 dimensions):")
print(baseline_embeddings[0][:8])
print("\nPYTORCH BASELINE GENERATED SUCCESSFULLY")


Note: you may need to restart the kernel to use updated packages.
Transformers version: 5.23.0
Loading model: sentence-transformers/all-MiniLM-L6-v2


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


========== BASELINE RESULTS ==========
Model: sentence-transformers/all-MiniLM-L6-v2
Embedding shape: (4, 384)
Inference time: 237.72 ms
Saved: /kaggle/working/experiment01/pytorch_embeddings.npy
Saved: /kaggle/working/experiment01/baseline_metadata.json
First embedding (first 8 dimensions):
[ 0.10148522  0.11179815  0.28715187 -0.02405886  0.2138843  -0.39157787
 -0.15980262  0.38504377]

PYTORCH BASELINE GENERATED SUCCESSFULLY


In [4]:

# EXPERIMENT 01 — Transformers Native ONNX Export

%pip install -q onnxruntime onnxscript

import time
import torch
import numpy as np
from pathlib import Path
from transformers.exporters import OnnxExporter, OnnxConfig, ExportedModel

OUTPUT_DIR = Path("/kaggle/working/experiment01")
EXPORT_DIR = OUTPUT_DIR / "onnx"

# 1. Export using Transformers 5.23.0 native exporter
print("Exporting PyTorch model to ONNX...")

exporter = OnnxExporter()
config = OnnxConfig(dynamic=True)

start = time.perf_counter()

exported = exporter.export(
    model,
    inputs,
    config=config
)

export_time = time.perf_counter() - start

# 2. Save standardized ExportedModel artifact
exported.save_pretrained(str(EXPORT_DIR))

print(f"\nExport completed in {export_time:.2f} seconds")
print("Exported artifact:", EXPORT_DIR)
print("Components:", list(exported.components.keys()))

# 3. Reload through the new ExportedModel interface
print("\nLoading exported model...")

onnx_model = ExportedModel.from_pretrained(
    str(EXPORT_DIR),
    device="cpu"
)

# 4. Run inference
with torch.inference_mode():
    start = time.perf_counter()

    onnx_outputs = onnx_model(**inputs)

    onnx_token_embeddings = onnx_outputs.last_hidden_state
    attention_mask = inputs["attention_mask"].unsqueeze(-1)

    onnx_embeddings = (
        (onnx_token_embeddings * attention_mask).sum(dim=1)
        / attention_mask.sum(dim=1).clamp(min=1)
    )

    onnx_embeddings = onnx_embeddings.cpu().numpy()
    onnx_time = time.perf_counter() - start

# 5. Save ONNX reference outputs
np.save(OUTPUT_DIR / "onnx_embeddings.npy", onnx_embeddings)

print("\n========== ONNX RESULTS ==========")
print("Output shape:", onnx_embeddings.shape)
print(f"Inference time: {onnx_time * 1000:.2f} ms")
print("Saved ONNX embeddings successfully.")

print("\nONNX EXPORT AND INFERENCE COMPLETE")


Note: you may need to restart the kernel to use updated packages.


[transformers] OnnxExporter is experimental and patches many backend internals; behaviour may differ from what was validated. Version drift detected — torch: installed 2.11.0, tested 2.12.0, onnx: installed 1.23.2, tested 1.21.0, onnxscript: installed 0.7.2, tested 0.7.0. If you hit issues, try the tested versions.


Exporting PyTorch model to ONNX...


`dynamic=True` with no explicit `dynamic_shapes` marks every input axis `Dim.AUTO`, so torch.export resolves symbolic shapes for all of them — including axes that are actually fixed (batch, a size-1 decode step, num_heads/head_dim). Passing explicit `dynamic_shapes` that mark only the axes which vary bypasses that symbolic-shape resolution and exports significantly faster.
W1114 17:39:19.171000 50 torch/onnx/_internal/exporter/_registration.py:86] Failed to infer the signature for function '<function _aten_bincount at 0x7b83aac30d60>' because 'name 'INT64' is not defined'All nodes targeting `aten.bincount.default` will be dispatched to this function
W1114 17:39:19.173000 50 torch/onnx/_internal/exporter/_registration.py:86] Failed to infer the signature for function '<function _aten_index_put at 0x7b83aac30cc0>' because 'name 'TReal' is not defined'All nodes targeting `aten.index_put.default` will be dispatched to this function
W1114 17:39:19.175000 50 torch/onnx/_internal/exporter/_re

[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅

Export completed in 13.59 seconds
Exported artifact: /kaggle/working/experiment01/onnx
Components: ['model']

Loading exported model...

========== ONNX RESULTS ==========
Output shape: (4, 384)
Inference time: 23.14 ms
Saved ONNX embeddings successfully.

ONNX EXPORT AND INFERENCE COMPLETE


In [5]:

# EXPERIMENT 01 — Final Benchmark & Correctness Evaluation

import numpy as np
import torch
import time
import json

ITERATIONS = 100
WARMUP = 10

def pytorch_inference():
    with torch.inference_mode():
        out = model(**inputs).last_hidden_state
        mask = inputs["attention_mask"].unsqueeze(-1)
        return (
            (out * mask).sum(dim=1)
            / mask.sum(dim=1).clamp(min=1)
        ).cpu().numpy()

def onnx_inference():
    with torch.inference_mode():
        out = onnx_model(**inputs).last_hidden_state
        mask = inputs["attention_mask"].unsqueeze(-1)
        return (
            (out * mask).sum(dim=1)
            / mask.sum(dim=1).clamp(min=1)
        ).cpu().numpy()

def benchmark(fn):
    for _ in range(WARMUP):
        fn()

    timings = []

    for _ in range(ITERATIONS):
        start = time.perf_counter()
        fn()
        timings.append((time.perf_counter() - start) * 1000)

    timings = np.array(timings)

    return {
        "mean_ms": float(np.mean(timings)),
        "median_ms": float(np.median(timings)),
        "p95_ms": float(np.percentile(timings, 95)),
        "throughput_texts_sec": float(
            len(texts) / (np.mean(timings) / 1000)
        )
    }

print("Benchmarking PyTorch...")
pytorch_results = benchmark(pytorch_inference)

print("Benchmarking ONNX Runtime...")
onnx_results = benchmark(onnx_inference)

# Numerical comparison using the latest inference outputs
pt_embeddings = pytorch_inference()
ort_embeddings = onnx_inference()

absolute_diff = np.abs(pt_embeddings - ort_embeddings)

cosine_scores = np.sum(
    pt_embeddings * ort_embeddings, axis=1
) / (
    np.linalg.norm(pt_embeddings, axis=1)
    * np.linalg.norm(ort_embeddings, axis=1)
)

speedup = (
    pytorch_results["mean_ms"] / onnx_results["mean_ms"]
)

# Save results
results = {
    "iterations": ITERATIONS,
    "warmup": WARMUP,
    "pytorch": pytorch_results,
    "onnx": onnx_results,
    "speedup": float(speedup),
    "max_absolute_difference": float(absolute_diff.max()),
    "mean_absolute_difference": float(absolute_diff.mean()),
    "mean_cosine_similarity": float(cosine_scores.mean()),
    "minimum_cosine_similarity": float(cosine_scores.min())
}

with open(OUTPUT_DIR / "benchmark_summary.json", "w") as f:
    json.dump(results, f, indent=2)

np.savetxt(
    OUTPUT_DIR / "benchmark_comparison.csv",
    np.array([
        [pytorch_results["mean_ms"], pytorch_results["median_ms"],
         pytorch_results["p95_ms"], pytorch_results["throughput_texts_sec"]],
        [onnx_results["mean_ms"], onnx_results["median_ms"],
         onnx_results["p95_ms"], onnx_results["throughput_texts_sec"]]
    ]),
    delimiter=",",
    header="mean_ms,median_ms,p95_ms,throughput_texts_sec",
    comments=""
)

# Final report
print("\n" + "=" * 60)
print("FINAL EXPERIMENT RESULTS")
print("=" * 60)

for name, result in [
    ("PYTORCH", pytorch_results),
    ("ONNX RUNTIME", onnx_results)
]:
    print(f"\n{name}")
    for metric, value in result.items():
        print(f"{metric}: {value:.4f}")

print("\nPERFORMANCE")
print(f"ONNX speedup: {speedup:.2f}x")

print("\nNUMERICAL CONSISTENCY")
print(f"Maximum absolute difference: {absolute_diff.max():.8f}")
print(f"Mean absolute difference: {absolute_diff.mean():.8f}")
print(f"Mean cosine similarity: {cosine_scores.mean():.8f}")
print(f"Minimum cosine similarity: {cosine_scores.min():.8f}")

print("\nSaved:")
print(OUTPUT_DIR / "benchmark_summary.json")
print(OUTPUT_DIR / "benchmark_comparison.csv")

print("\nEXPERIMENT BENCHMARK COMPLETE")


Benchmarking PyTorch...
Benchmarking ONNX Runtime...

FINAL EXPERIMENT RESULTS

PYTORCH
mean_ms: 23.2762
median_ms: 22.9392
p95_ms: 25.2549
throughput_texts_sec: 171.8495

ONNX RUNTIME
mean_ms: 19.2464
median_ms: 19.0132
p95_ms: 20.3981
throughput_texts_sec: 207.8307

PERFORMANCE
ONNX speedup: 1.21x

NUMERICAL CONSISTENCY
Maximum absolute difference: 0.00000088
Mean absolute difference: 0.00000020
Mean cosine similarity: 0.99999994
Minimum cosine similarity: 0.99999988

Saved:
/kaggle/working/experiment01/benchmark_summary.json
/kaggle/working/experiment01/benchmark_comparison.csv

EXPERIMENT BENCHMARK COMPLETE


In [6]:

# EXPERIMENT 01 — Standard ONNX vs CPU-Fused ONNX

import time
import numpy as np
import onnx
from pathlib import Path
from transformers.exporters import OnnxExporter, OnnxConfig, ExportedModel

FUSED_DIR = OUTPUT_DIR / "onnx_cpu_fused"
FUSED_DIR.mkdir(exist_ok=True)

# 1. Export with CPU-specific ONNX Runtime fusions
print("Exporting CPU-fused ONNX model...")

fused_export = OnnxExporter().export(
    model,
    inputs,
    config=OnnxConfig(dynamic=True, ort_fusions="cpu")
)

fused_export.save_pretrained(str(FUSED_DIR))

# 2. Load exported model
fused_model = ExportedModel.from_pretrained(
    str(FUSED_DIR),
    device="cpu"
)

print("CPU-fused export loaded.")

# 3. Inspect actual ONNX graph operators
onnx_file = next(FUSED_DIR.glob("*.onnx"))
graph = onnx.load(str(onnx_file), load_external_data=False)

domains = {}
for node in graph.graph.node:
    domains[node.domain or "ai.onnx"] = (
        domains.get(node.domain or "ai.onnx", 0) + 1
    )

print("\nONNX graph operator domains:")
print(domains)

# 4. Inference function
def fused_inference():
    with torch.inference_mode():
        out = fused_model(**inputs).last_hidden_state
        mask = inputs["attention_mask"].unsqueeze(-1)
        return (
            (out * mask).sum(dim=1)
            / mask.sum(dim=1).clamp(min=1)
        ).cpu().numpy()

# 5. Correctness comparison
fused_embeddings = fused_inference()
standard_embeddings = onnx_inference()

diff = np.abs(fused_embeddings - standard_embeddings)

cosine = np.sum(
    fused_embeddings * standard_embeddings, axis=1
) / (
    np.linalg.norm(fused_embeddings, axis=1)
    * np.linalg.norm(standard_embeddings, axis=1)
)

# 6. Benchmark
def measure(fn, warmup=10, iterations=100):
    for _ in range(warmup):
        fn()

    times = []
    for _ in range(iterations):
        start = time.perf_counter()
        fn()
        times.append((time.perf_counter() - start) * 1000)

    return {
        "mean_ms": float(np.mean(times)),
        "median_ms": float(np.median(times)),
        "p95_ms": float(np.percentile(times, 95))
    }

print("\nBenchmarking standard ONNX...")
standard_bench = measure(onnx_inference)

print("Benchmarking CPU-fused ONNX...")
fused_bench = measure(fused_inference)

print("\n========== FINAL FUSION RESULTS ==========")
print("Standard ONNX:", standard_bench)
print("CPU-fused ONNX:", fused_bench)

print(
    "\nFusion speedup:",
    round(standard_bench["mean_ms"] / fused_bench["mean_ms"], 3),
    "x"
)

print("\n========== OUTPUT CONSISTENCY ==========")
print("Maximum absolute difference:", diff.max())
print("Mean absolute difference:", diff.mean())
print("Mean cosine similarity:", cosine.mean())
print("Minimum cosine similarity:", cosine.min())

print("\n========== EXPERIMENT COMPLETE ==========")


[transformers] OnnxExporter is experimental and patches many backend internals; behaviour may differ from what was validated. Version drift detected — torch: installed 2.11.0, tested 2.12.0, onnx: installed 1.23.2, tested 1.21.0, onnxscript: installed 0.7.2, tested 0.7.0. If you hit issues, try the tested versions.


Exporting CPU-fused ONNX model...


W1114 17:44:50.227000 50 torch/onnx/_internal/exporter/_registration.py:86] Failed to infer the signature for function '<function _aten_bincount at 0x7b83aac30d60>' because 'name 'INT64' is not defined'All nodes targeting `aten.bincount.default` will be dispatched to this function
W1114 17:44:50.228000 50 torch/onnx/_internal/exporter/_registration.py:86] Failed to infer the signature for function '<function _aten_index_put at 0x7b83aac30cc0>' because 'name 'TReal' is not defined'All nodes targeting `aten.index_put.default` will be dispatched to this function
W1114 17:44:50.229000 50 torch/onnx/_internal/exporter/_registration.py:86] Failed to infer the signature for function '<function _aten_grouped_mm at 0x7b83aac30ea0>' because 'name 'TReal' is not defined'All nodes targeting `aten._grouped_mm.default` will be dispatched to this function
W1114 17:44:50.230000 50 torch/onnx/_internal/exporter/_registration.py:86] Failed to infer the signature for function '<function _aten_grouped_mm 

[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
CPU-fused export loaded.

ONNX graph operator domains:
{'ai.onnx': 224, 'com.microsoft': 18}

Benchmarking standard ONNX...
Benchmarking CPU-fused ONNX...

========== FINAL FUSION RESULTS ==========
Standard ONNX: {'mean_ms': 19.49625547000096, 'median_ms': 19.307092000019566, 'p95_ms': 21.0918241499769}
CPU-fused ONNX: {'mean_ms': 18.84305218000577, 'median_ms': 18.67638800001714, 'p95_ms': 19.989588100071387}

Fusion speedup: 1.035 x

========== OUTPUT CONSISTENCY ==========
Maximum absolute difference: 5.662441e-07
Mean absolute difference: 1.233396e-07
Mean cosine similarity: 1.0
Minimum cosine similarity: 0.9999999

========== EXPERIMENT COMPLETE ==========


In [7]:

# STEP 1: Inspect the intended Sentence Transformers pipeline
# This downloads only small configuration files if they are not cached.
# It does NOT download model weights or modify existing models.

from huggingface_hub import hf_hub_download
from pathlib import Path
import json

REPO_ID = "sentence-transformers/all-MiniLM-L6-v2"

files_to_inspect = [
    "modules.json",
    "config_sentence_transformers.json",
    "1_Pooling/config.json",
]

print("=" * 65)
print("MODEL CONFIGURATION INSPECTION")
print("=" * 65)

print("\nLoaded PyTorch model:")
print("Model path:", model.config._name_or_path)
print("Hidden size:", model.config.hidden_size)

for filename in files_to_inspect:
    print(f"\n{'-' * 65}")
    print(f"FILE: {filename}")
    print("-" * 65)

    try:
        local_path = hf_hub_download(
            repo_id=REPO_ID,
            filename=filename
        )

        with open(local_path, "r", encoding="utf-8") as f:
            config_data = json.load(f)

        print(json.dumps(config_data, indent=2))

    except Exception as e:
        print(f"Could not retrieve this file: {e}")

print("\n" + "=" * 65)
print("INSPECTION COMPLETE")
print("=" * 65)


MODEL CONFIGURATION INSPECTION

Loaded PyTorch model:
Model path: sentence-transformers/all-MiniLM-L6-v2
Hidden size: 384

-----------------------------------------------------------------
FILE: modules.json
-----------------------------------------------------------------


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

[
  {
    "idx": 0,
    "name": "0",
    "path": "",
    "type": "sentence_transformers.models.Transformer"
  },
  {
    "idx": 1,
    "name": "1",
    "path": "1_Pooling",
    "type": "sentence_transformers.models.Pooling"
  },
  {
    "idx": 2,
    "name": "2",
    "path": "2_Normalize",
    "type": "sentence_transformers.models.Normalize"
  }
]

-----------------------------------------------------------------
FILE: config_sentence_transformers.json
-----------------------------------------------------------------


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

{
  "__version__": {
    "sentence_transformers": "2.0.0",
    "transformers": "4.6.1",
    "pytorch": "1.8.1"
  }
}

-----------------------------------------------------------------
FILE: 1_Pooling/config.json
-----------------------------------------------------------------


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

{
  "word_embedding_dimension": 384,
  "pooling_mode_cls_token": false,
  "pooling_mode_mean_tokens": true,
  "pooling_mode_max_tokens": false,
  "pooling_mode_mean_sqrt_len_tokens": false
}

INSPECTION COMPLETE


In [8]:

# STEP 2: Validate manual embeddings against canonical pipeline

import torch
import torch.nn.functional as F
import numpy as np

from sentence_transformers import SentenceTransformer

MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"

# Reuse the original input texts and loaded PyTorch model/tokenizer.
assert "texts" in globals(), "Original texts variable not found."
assert "model" in globals(), "Original PyTorch model not found."
assert "tokenizer" in globals(), "Original tokenizer not found."

# Reproduce our original manual implementation.
inputs = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

model.eval()

with torch.no_grad():
    outputs = model(**inputs)
    token_embeddings = outputs.last_hidden_state

    mask = inputs["attention_mask"].unsqueeze(-1).to(
        token_embeddings.dtype
    )

    summed = (token_embeddings * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1)

    manual_mean = summed / counts

# The intended model pipeline includes L2 normalization.
manual_normalized = F.normalize(manual_mean, p=2, dim=1)

# Load canonical Sentence Transformers implementation.
# This loads the same model, but does not replace our original model.
canonical_model = SentenceTransformer(
    MODEL_ID,
    device="cpu"
)

canonical_embeddings = canonical_model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=False,
    show_progress_bar=False
)

canonical_normalized = canonical_model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False
)

# Compare implementations.
def compare_embeddings(a, b, label):
    diff = np.abs(a - b)

    a_norm = a / np.maximum(
        np.linalg.norm(a, axis=1, keepdims=True), 1e-12
    )
    b_norm = b / np.maximum(
        np.linalg.norm(b, axis=1, keepdims=True), 1e-12
    )

    cosine = np.sum(a_norm * b_norm, axis=1)

    print(f"\n{label}")
    print("Shape:", a.shape)
    print("Max absolute error:", float(diff.max()))
    print("Mean absolute error:", float(diff.mean()))
    print("Mean cosine similarity:", float(cosine.mean()))
    print("Minimum cosine similarity:", float(cosine.min()))

manual_mean_np = manual_mean.numpy()
manual_normalized_np = manual_normalized.numpy()

compare_embeddings(
    manual_mean_np,
    canonical_embeddings,
    "MANUAL MEAN vs CANONICAL MEAN"
)

compare_embeddings(
    manual_normalized_np,
    canonical_normalized,
    "MANUAL NORMALIZED vs CANONICAL NORMALIZED"
)

print("\nEmbedding norms:")
print("Manual mean:", np.linalg.norm(manual_mean_np, axis=1))
print("Manual normalized:", np.linalg.norm(manual_normalized_np, axis=1))
print("Canonical normalized:", np.linalg.norm(canonical_normalized, axis=1))


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


MANUAL MEAN vs CANONICAL MEAN
Shape: (4, 384)
Max absolute error: 0.7958359122276306
Mean absolute error: 0.18779774010181427
Mean cosine similarity: 1.0
Minimum cosine similarity: 1.0

MANUAL NORMALIZED vs CANONICAL NORMALIZED
Shape: (4, 384)
Max absolute error: 1.4901161193847656e-08
Mean absolute error: 2.6048081469554063e-09
Mean cosine similarity: 1.0000001192092896
Minimum cosine similarity: 1.0

Embedding norms:
Manual mean: [6.121807  6.1711884 4.9749804 5.399025 ]
Manual normalized: [0.99999994 0.99999994 0.99999994 1.        ]
Canonical normalized: [0.99999994 1.         1.         1.        ]


In [9]:

# STEP 3: PyTorch vs ONNX normalized embedding and retrieval consistency
#
# Purpose:
# 1. Generate normalized embeddings using both execution paths.
# 2. Compare numerical agreement.
# 3. Compare cosine-similarity rankings.
# 4. Save reproducible results.
#
# Does NOT re-export models or install packages.

import os
import json
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from scipy.stats import spearmanr

# ---------------------------------------------------------
# 1. Validate existing notebook state
# ---------------------------------------------------------

required_objects = ["model", "tokenizer", "onnx_model", "texts"]

missing = [
    name for name in required_objects
    if name not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing notebook objects: {missing}. "
        "Do not re-export yet; inspect notebook state first."
    )

assert len(texts) >= 2, "At least two texts are required."

model.eval()

# ---------------------------------------------------------
# 2. Generate normalized PyTorch embeddings
# ---------------------------------------------------------

inputs = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

with torch.inference_mode():
    torch_outputs = model(**inputs)

    token_embeddings = torch_outputs.last_hidden_state

    attention_mask = inputs["attention_mask"].unsqueeze(-1).to(
        token_embeddings.dtype
    )

    summed_embeddings = (token_embeddings * attention_mask).sum(dim=1)

    token_counts = attention_mask.sum(dim=1).clamp(min=1)

    mean_embeddings = summed_embeddings / token_counts

    pytorch_embeddings = F.normalize(
        mean_embeddings,
        p=2,
        dim=1
    ).cpu().numpy()

# ---------------------------------------------------------
# 3. Generate normalized ONNX embeddings
# ---------------------------------------------------------
# Use the existing ONNX model and same tokenized inputs.
# The exporter returns model outputs; pooling and normalization
# are applied identically after inference.

onnx_inputs = {
    "input_ids": inputs["input_ids"].cpu().numpy(),
    "attention_mask": inputs["attention_mask"].cpu().numpy(),
}

with torch.inference_mode():
    onnx_outputs = onnx_model(**inputs)

# Handle the model output without assuming its concrete class.
if hasattr(onnx_outputs, "last_hidden_state"):
    onnx_token_embeddings = onnx_outputs.last_hidden_state
elif isinstance(onnx_outputs, dict):
    onnx_token_embeddings = onnx_outputs["last_hidden_state"]
elif isinstance(onnx_outputs, (tuple, list)):
    onnx_token_embeddings = onnx_outputs[0]
else:
    raise TypeError(
        f"Unexpected ONNX output type: {type(onnx_outputs)}"
    )

if isinstance(onnx_token_embeddings, torch.Tensor):
    onnx_token_embeddings = onnx_token_embeddings.detach().cpu().numpy()

onnx_mask = onnx_inputs["attention_mask"][..., None].astype(
    onnx_token_embeddings.dtype
)

onnx_sum = (onnx_token_embeddings * onnx_mask).sum(axis=1)

onnx_count = np.maximum(onnx_mask.sum(axis=1), 1)

onnx_mean = onnx_sum / onnx_count

onnx_embeddings = onnx_mean / np.maximum(
    np.linalg.norm(onnx_mean, axis=1, keepdims=True),
    1e-12
)

# ---------------------------------------------------------
# 4. Numerical comparison
# ---------------------------------------------------------

difference = np.abs(
    pytorch_embeddings - onnx_embeddings
)

pytorch_norms = np.linalg.norm(
    pytorch_embeddings, axis=1
)

onnx_norms = np.linalg.norm(
    onnx_embeddings, axis=1
)

cosine_per_text = np.sum(
    pytorch_embeddings * onnx_embeddings,
    axis=1
)

print("=" * 65)
print("NORMALIZED EMBEDDING COMPARISON")
print("=" * 65)

print("PyTorch shape:", pytorch_embeddings.shape)
print("ONNX shape:", onnx_embeddings.shape)

print("Maximum absolute error:", float(difference.max()))
print("Mean absolute error:", float(difference.mean()))

print("Mean cosine similarity:", float(cosine_per_text.mean()))
print("Minimum cosine similarity:", float(cosine_per_text.min()))

print("PyTorch embedding norms:", pytorch_norms)
print("ONNX embedding norms:", onnx_norms)

# ---------------------------------------------------------
# 5. Retrieval ranking consistency
# ---------------------------------------------------------

# Each text acts as a query against the same four-text collection.
# Self-match is excluded from ranking.

pytorch_similarity = pytorch_embeddings @ pytorch_embeddings.T
onnx_similarity = onnx_embeddings @ onnx_embeddings.T

np.fill_diagonal(pytorch_similarity, -np.inf)
np.fill_diagonal(onnx_similarity, -np.inf)

n = len(texts)
k = min(3, n - 1)

topk_overlap = []
rank_correlations = []
ranking_changes = []

for query_idx in range(n):

    pytorch_ranking = np.argsort(
        -pytorch_similarity[query_idx]
    )[:k]

    onnx_ranking = np.argsort(
        -onnx_similarity[query_idx]
    )[:k]

    overlap = (
        len(set(pytorch_ranking) & set(onnx_ranking)) / k
    )

    topk_overlap.append(overlap)

    # Compare full candidate rankings, excluding the query itself.
    candidates = [
        idx for idx in range(n)
        if idx != query_idx
    ]

    pytorch_order = sorted(
        candidates,
        key=lambda idx: -pytorch_similarity[query_idx, idx]
    )

    onnx_order = sorted(
        candidates,
        key=lambda idx: -onnx_similarity[query_idx, idx]
    )

    pytorch_ranks = [
        pytorch_order.index(idx) for idx in candidates
    ]

    onnx_ranks = [
        onnx_order.index(idx) for idx in candidates
    ]

    correlation = spearmanr(
        pytorch_ranks,
        onnx_ranks
    ).statistic

    rank_correlations.append(float(correlation))

    if pytorch_order != onnx_order:
        ranking_changes.append(query_idx)

    print(f"\nQuery {query_idx}: {texts[query_idx][:80]}")
    print("PyTorch top-k:", pytorch_ranking.tolist())
    print("ONNX top-k:   ", onnx_ranking.tolist())
    print(f"Top-k overlap: {overlap:.2%}")
    print(f"Rank correlation: {correlation:.4f}")

# ---------------------------------------------------------
# 6. Summary and save results
# ---------------------------------------------------------

summary = {
    "model": "sentence-transformers/all-MiniLM-L6-v2",
    "number_of_texts": n,
    "embedding_dimension": int(pytorch_embeddings.shape[1]),
    "normalization": "L2",
    "top_k": k,
    "max_absolute_error": float(difference.max()),
    "mean_absolute_error": float(difference.mean()),
    "mean_cosine_similarity": float(cosine_per_text.mean()),
    "minimum_cosine_similarity": float(cosine_per_text.min()),
    "mean_top_k_overlap": float(np.mean(topk_overlap)),
    "mean_rank_correlation": float(np.mean(rank_correlations)),
    "ranking_changes": ranking_changes,
    "pytorch_embedding_norms": pytorch_norms.tolist(),
    "onnx_embedding_norms": onnx_norms.tolist(),
}

output_dir = Path("/kaggle/working/experiment01")
output_dir.mkdir(parents=True, exist_ok=True)

np.save(
    output_dir / "pytorch_normalized_embeddings.npy",
    pytorch_embeddings
)

np.save(
    output_dir / "onnx_normalized_embeddings.npy",
    onnx_embeddings
)

with open(
    output_dir / "retrieval_consistency.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 65)
print("FINAL RETRIEVAL CONSISTENCY SUMMARY")
print("=" * 65)

print(json.dumps(summary, indent=2))
print("\nSaved results to:", output_dir)


NORMALIZED EMBEDDING COMPARISON
PyTorch shape: (4, 384)
ONNX shape: (4, 384)
Maximum absolute error: 1.6391277313232422e-07
Mean absolute error: 3.4775698765088237e-08
Mean cosine similarity: 1.0
Minimum cosine similarity: 0.9999998807907104
PyTorch embedding norms: [0.99999994 0.99999994 0.99999994 1.        ]
ONNX embedding norms: [1.         0.99999994 1.         0.99999994]

Query 0: Artificial intelligence is transforming software development.
PyTorch top-k: [1, 3, 2]
ONNX top-k:    [1, 3, 2]
Top-k overlap: 100.00%
Rank correlation: 1.0000

Query 1: Machine learning models require efficient inference for production.
PyTorch top-k: [3, 0, 2]
ONNX top-k:    [3, 0, 2]
Top-k overlap: 100.00%
Rank correlation: 1.0000

Query 2: Hugging Face provides tools for loading and deploying pretrained models.
PyTorch top-k: [3, 1, 0]
ONNX top-k:    [3, 1, 0]
Top-k overlap: 100.00%
Rank correlation: 1.0000

Query 3: ONNX Runtime can execute models exported from PyTorch.
PyTorch top-k: [2, 1, 0]
ON

In [10]:

# STEP 4: Controlled CPU inference benchmark
#
# PyTorch vs Standard ONNX Runtime vs CPU-fused ONNX Runtime
#
# 5 rounds | 10 warmups | 100 measurements per backend per round
# Explicit CPU thread settings | Randomized execution order
#
# No installation, no re-exporting, no model modifications.

import os
import time
import random
import json
import csv
import numpy as np
import torch
import onnx
import onnxruntime as ort

from pathlib import Path
from collections import Counter

# ---------------------------------------------------------
# 1. Validate existing objects
# ---------------------------------------------------------

assert "model" in globals(), "PyTorch model missing."
assert "tokenizer" in globals(), "Tokenizer missing."
assert "texts" in globals(), "Original texts missing."

model.eval()

# ---------------------------------------------------------
# 2. Benchmark configuration
# ---------------------------------------------------------

NUM_THREADS = 4
NUM_WARMUPS = 10
NUM_ITERATIONS = 100
NUM_ROUNDS = 5

SEED = 42
rng = random.Random(SEED)

# ---------------------------------------------------------
# 3. Configure PyTorch threads
# ---------------------------------------------------------

torch.set_num_threads(NUM_THREADS)

try:
    torch.set_num_interop_threads(1)
    interop_status = "1 inter-op thread configured"
except RuntimeError as e:
    interop_status = f"Could not change inter-op setting: {e}"

print("=" * 70)
print("BENCHMARK CONFIGURATION")
print("=" * 70)

print("CPU logical cores:", os.cpu_count())
print("PyTorch intra-op threads:", torch.get_num_threads())
print("PyTorch inter-op:", interop_status)
print("ONNX Runtime version:", ort.__version__)
print("Warmups:", NUM_WARMUPS)
print("Iterations per round:", NUM_ITERATIONS)
print("Rounds:", NUM_ROUNDS)

# ---------------------------------------------------------
# 4. Locate existing ONNX exports
# ---------------------------------------------------------

search_root = Path("/kaggle/working")

onnx_files = [
    p for p in search_root.rglob("*.onnx")
    if p.is_file()
]

if not onnx_files:
    raise FileNotFoundError(
        "No saved ONNX files found under /kaggle/working."
    )

graph_info = []

for path in onnx_files:
    try:
        graph = onnx.load(str(path), load_external_data=False)

        microsoft_nodes = sum(
            node.domain == "com.microsoft"
            for node in graph.graph.node
        )

        graph_info.append({
            "path": path,
            "microsoft_nodes": microsoft_nodes,
            "total_nodes": len(graph.graph.node),
        })

    except Exception as e:
        print("Skipping unreadable ONNX file:", path, e)

standard_candidates = [
    item for item in graph_info
    if item["microsoft_nodes"] == 0
]

fused_candidates = [
    item for item in graph_info
    if item["microsoft_nodes"] > 0
]

if not standard_candidates or not fused_candidates:
    print("\nDiscovered ONNX files:")
    for item in graph_info:
        print(item)

    raise RuntimeError(
        "Could not identify both standard and CPU-fused ONNX exports. "
        "Share the discovered file list before proceeding."
    )

# Select one standard and one fused export.
# Prefer paths containing useful identifying names.
standard_candidates.sort(
    key=lambda x: (
        "fused" in str(x["path"]).lower(),
        len(str(x["path"]))
    )
)

fused_candidates.sort(
    key=lambda x: (
        "fused" not in str(x["path"]).lower(),
        -x["microsoft_nodes"]
    )
)

standard_path = standard_candidates[0]["path"]
fused_path = fused_candidates[0]["path"]

print("\nSelected ONNX models:")
print("Standard:", standard_path)
print("Fused:", fused_path)

print(
    "Standard graph nodes:",
    standard_candidates[0]["total_nodes"]
)
print(
    "Fused graph nodes:",
    fused_candidates[0]["total_nodes"]
)
print(
    "Fused Microsoft-domain nodes:",
    fused_candidates[0]["microsoft_nodes"]
)

# ---------------------------------------------------------
# 5. Create controlled ONNX Runtime sessions
# ---------------------------------------------------------

def create_ort_session(model_path):

    options = ort.SessionOptions()

    options.intra_op_num_threads = NUM_THREADS
    options.inter_op_num_threads = 1

    options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL

    options.graph_optimization_level = (
        ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    )

    session = ort.InferenceSession(
        str(model_path),
        sess_options=options,
        providers=["CPUExecutionProvider"]
    )

    return session


standard_session = create_ort_session(standard_path)
fused_session = create_ort_session(fused_path)

print("\nONNX Runtime session configuration:")
print("Standard providers:", standard_session.get_providers())
print("Fused providers:", fused_session.get_providers())
print("Intra-op threads:", NUM_THREADS)
print("Inter-op threads: 1")
print("Execution mode: Sequential")
print("Graph optimization: ORT_ENABLE_ALL")

# ---------------------------------------------------------
# 6. Prepare identical input data
# ---------------------------------------------------------

benchmark_inputs = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

with torch.inference_mode():
    pytorch_call = lambda: model(**benchmark_inputs)

def make_ort_feed(session):

    feed = {}

    for input_meta in session.get_inputs():

        name = input_meta.name

        if name not in benchmark_inputs:
            raise ValueError(
                f"Required ONNX input '{name}' "
                "is not available in tokenizer output."
            )

        feed[name] = (
            benchmark_inputs[name]
            .cpu()
            .numpy()
        )

    return feed


standard_feed = make_ort_feed(standard_session)
fused_feed = make_ort_feed(fused_session)

def standard_call():
    return standard_session.run(None, standard_feed)

def fused_call():
    return fused_session.run(None, fused_feed)

runners = {
    "pytorch": pytorch_call,
    "onnx_standard": standard_call,
    "onnx_fused": fused_call,
}

# ---------------------------------------------------------
# 7. Warm up all execution paths
# ---------------------------------------------------------

print("\nWarming up execution paths...")

with torch.inference_mode():

    for name, runner in runners.items():

        for _ in range(NUM_WARMUPS):
            runner()

        print(f"{name}: {NUM_WARMUPS} warmups complete")

# ---------------------------------------------------------
# 8. Run randomized benchmark rounds
# ---------------------------------------------------------

measurements = []

print("\nStarting benchmark...")
print("=" * 70)

for round_number in range(1, NUM_ROUNDS + 1):

    round_measurements = {
        name: [] for name in runners
    }

    for iteration in range(NUM_ITERATIONS):

        execution_order = list(runners.keys())
        rng.shuffle(execution_order)

        for backend in execution_order:

            start = time.perf_counter_ns()

            with torch.inference_mode():
                runners[backend]()

            elapsed_ms = (
                time.perf_counter_ns() - start
            ) / 1_000_000

            round_measurements[backend].append(elapsed_ms)

            measurements.append({
                "round": round_number,
                "iteration": iteration + 1,
                "backend": backend,
                "latency_ms": elapsed_ms,
            })

    print(f"\nRound {round_number}/{NUM_ROUNDS}")

    for backend, values in round_measurements.items():

        median = float(np.median(values))

        print(
            f"{backend:15s} "
            f"median={median:.4f} ms | "
            f"IQR={np.percentile(values, 75) - np.percentile(values, 25):.4f} ms"
        )

# ---------------------------------------------------------
# 9. Aggregate results
# ---------------------------------------------------------

summary = {}

for backend in runners:

    values = np.array([
        row["latency_ms"]
        for row in measurements
        if row["backend"] == backend
    ])

    per_round_medians = []

    for round_number in range(1, NUM_ROUNDS + 1):

        round_values = [
            row["latency_ms"]
            for row in measurements
            if (
                row["backend"] == backend
                and row["round"] == round_number
            )
        ]

        per_round_medians.append(
            float(np.median(round_values))
        )

    summary[backend] = {
        "measurements": len(values),
        "overall_median_ms": float(np.median(values)),
        "mean_ms": float(np.mean(values)),
        "p95_ms": float(np.percentile(values, 95)),
        "iqr_ms": float(
            np.percentile(values, 75)
            - np.percentile(values, 25)
        ),
        "round_medians_ms": per_round_medians,
        "round_median_mean_ms": float(
            np.mean(per_round_medians)
        ),
        "round_median_std_ms": float(
            np.std(per_round_medians, ddof=1)
        ),
    }

# ---------------------------------------------------------
# 10. Calculate relative speedups
# ---------------------------------------------------------

pytorch_median = summary["pytorch"]["overall_median_ms"]
standard_median = summary["onnx_standard"]["overall_median_ms"]
fused_median = summary["onnx_fused"]["overall_median_ms"]

speedups = {
    "standard_onnx_vs_pytorch_percent": (
        (pytorch_median / standard_median) - 1
    ) * 100,

    "fused_vs_standard_percent": (
        (standard_median / fused_median) - 1
    ) * 100,

    "fused_vs_pytorch_percent": (
        (pytorch_median / fused_median) - 1
    ) * 100,
}

# ---------------------------------------------------------
# 11. Save raw measurements and summary
# ---------------------------------------------------------

output_dir = Path("/kaggle/working/experiment01")
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "controlled_benchmark_raw.csv"

with open(csv_path, "w", newline="", encoding="utf-8") as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "round",
            "iteration",
            "backend",
            "latency_ms",
        ]
    )

    writer.writeheader()
    writer.writerows(measurements)

benchmark_summary = {
    "benchmark_type": "controlled repeated measurement rounds",
    "independent_process_sessions": False,
    "rounds": NUM_ROUNDS,
    "warmups_per_backend": NUM_WARMUPS,
    "iterations_per_backend_per_round": NUM_ITERATIONS,
    "pytorch_threads": NUM_THREADS,
    "onnx_intra_op_threads": NUM_THREADS,
    "onnx_inter_op_threads": 1,
    "onnx_execution_mode": "ORT_SEQUENTIAL",
    "hardware_logical_cores": os.cpu_count(),
    "onnxruntime_version": ort.__version__,
    "results": summary,
    "speedups_percent": speedups,
}

json_path = output_dir / "controlled_benchmark_summary.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(benchmark_summary, f, indent=2)

# ---------------------------------------------------------
# 12. Final report
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("CONTROLLED BENCHMARK RESULTS")
print("=" * 70)

for backend, result in summary.items():

    print(f"\n{backend.upper()}")

    print(f"Median: {result['overall_median_ms']:.4f} ms")
    print(f"Mean:   {result['mean_ms']:.4f} ms")
    print(f"P95:    {result['p95_ms']:.4f} ms")
    print(f"IQR:    {result['iqr_ms']:.4f} ms")

    print(
        "Round medians:",
        [
            round(x, 4)
            for x in result["round_medians_ms"]
        ]
    )

print("\nSPEEDUP COMPARISON")

for name, value in speedups.items():
    print(f"{name}: {value:+.2f}%")

print("\nSaved raw measurements:", csv_path)
print("Saved summary:", json_path)

print("\nBENCHMARK COMPLETE")
print("=" * 70)


BENCHMARK CONFIGURATION
CPU logical cores: 4
PyTorch intra-op threads: 4
PyTorch inter-op: 1 inter-op thread configured
ONNX Runtime version: 1.32.0
Warmups: 10
Iterations per round: 100
Rounds: 5

Selected ONNX models:
Standard: /kaggle/working/experiment01/onnx/model.onnx
Fused: /kaggle/working/experiment01/onnx_cpu_fused/model.onnx
Standard graph nodes: 287
Fused graph nodes: 242
Fused Microsoft-domain nodes: 18

ONNX Runtime session configuration:
Standard providers: ['CPUExecutionProvider']
Fused providers: ['CPUExecutionProvider']
Intra-op threads: 4
Inter-op threads: 1
Execution mode: Sequential
Graph optimization: ORT_ENABLE_ALL

Warming up execution paths...
pytorch: 10 warmups complete
onnx_standard: 10 warmups complete
onnx_fused: 10 warmups complete

Starting benchmark...

Round 1/5
pytorch         median=169.2365 ms | IQR=50.0382 ms
onnx_standard   median=45.6846 ms | IQR=40.8031 ms
onnx_fused      median=46.5460 ms | IQR=42.1611 ms

Round 2/5
pytorch         median=174.17

In [11]:

from pathlib import Path
from collections import Counter
import json
import onnx

BASE_DIR = Path("/kaggle/working/experiment01")

MODEL_PATHS = {
    "standard": BASE_DIR / "onnx" / "model.onnx",
    "cpu_fused": BASE_DIR / "onnx_cpu_fused" / "model.onnx",
}

def inspect_model(name, path):
    if not path.exists():
        raise FileNotFoundError(f"{name} model not found: {path}")

    model = onnx.load(str(path), load_external_data=False)
    graph = model.graph

    domain_counts = Counter(
        node.domain if node.domain else "ai.onnx"
        for node in graph.node
    )

    operator_counts = Counter(
        f"{node.domain if node.domain else 'ai.onnx'}::{node.op_type}"
        for node in graph.node
    )

    microsoft_ops = {
        key: count
        for key, count in sorted(operator_counts.items())
        if key.startswith("com.microsoft::")
    }

    return {
        "model_path": str(path),
        "file_size_mb": round(path.stat().st_size / (1024 ** 2), 3),
        "total_nodes": len(graph.node),
        "domain_counts": dict(sorted(domain_counts.items())),
        "operator_counts": dict(sorted(operator_counts.items())),
        "microsoft_operators": microsoft_ops,
        "opset_imports": [
            {"domain": item.domain or "ai.onnx", "version": item.version}
            for item in model.opset_import
        ],
        "inputs": [
            {"name": item.name}
            for item in graph.input
        ],
        "outputs": [
            {"name": item.name}
            for item in graph.output
        ],
    }

results = {
    name: inspect_model(name, path)
    for name, path in MODEL_PATHS.items()
}

for name, result in results.items():
    print("\n" + "=" * 65)
    print(name.upper())
    print("=" * 65)
    print(f"File size: {result['file_size_mb']} MB")
    print(f"Total nodes: {result['total_nodes']}")

    print("\nDomain counts:")
    for domain, count in result["domain_counts"].items():
        print(f"  {domain}: {count}")

    print("\nMicrosoft operators:")
    if result["microsoft_operators"]:
        for op, count in result["microsoft_operators"].items():
            print(f"  {op}: {count}")
    else:
        print("  None")

    print("\nMost frequent operators:")
    for op, count in sorted(
        result["operator_counts"].items(),
        key=lambda item: item[1],
        reverse=True
    )[:15]:
        print(f"  {op}: {count}")

output_path = BASE_DIR / "graph_inspection.json"
with open(output_path, "w") as f:
    json.dump(results, f, indent=2)

print("\nSaved graph inspection:", output_path)



STANDARD
File size: 0.581 MB
Total nodes: 287

Domain counts:
  ai.onnx: 287

Microsoft operators:
  None

Most frequent operators:
  ai.onnx::Add: 56
  ai.onnx::MatMul: 48
  ai.onnx::Reshape: 36
  ai.onnx::Transpose: 30
  ai.onnx::Slice: 19
  ai.onnx::Concat: 15
  ai.onnx::LayerNormalization: 13
  ai.onnx::Mul: 13
  ai.onnx::Shape: 9
  ai.onnx::Unsqueeze: 7
  ai.onnx::Gelu: 6
  ai.onnx::Softmax: 6
  ai.onnx::Where: 6
  ai.onnx::Gather: 4
  ai.onnx::Cast: 3

CPU_FUSED
File size: 0.307 MB
Total nodes: 242

Domain counts:
  ai.onnx: 224
  com.microsoft: 18

Microsoft operators:
  com.microsoft::BiasGelu: 6
  com.microsoft::SkipLayerNormalization: 12

Most frequent operators:
  ai.onnx::Reshape: 72
  ai.onnx::MatMul: 36
  ai.onnx::Add: 26
  ai.onnx::Transpose: 25
  com.microsoft::SkipLayerNormalization: 12
  ai.onnx::Mul: 7
  ai.onnx::Unsqueeze: 7
  ai.onnx::Softmax: 6
  ai.onnx::Split: 6
  ai.onnx::Where: 6
  com.microsoft::BiasGelu: 6
  ai.onnx::Concat: 4
  ai.onnx::Gather: 4
  ai.onnx

In [12]:

from pathlib import Path
import json
from datetime import datetime, timezone

BASE_DIR = Path("/kaggle/working/experiment01")
REPORT_PATH = BASE_DIR / "final_experiment_report.json"

# Collect saved JSON artifacts.
artifacts = {}

for path in sorted(BASE_DIR.rglob("*.json")):
    if path == REPORT_PATH:
        continue

    try:
        with open(path, "r") as f:
            artifacts[str(path.relative_to(BASE_DIR))] = json.load(f)
    except Exception as exc:
        print(f"Could not read {path.name}: {exc}")

# Collect CSV artifact metadata without duplicating raw measurements.
csv_artifacts = []

for path in sorted(BASE_DIR.rglob("*.csv")):
    csv_artifacts.append({
        "path": str(path.relative_to(BASE_DIR)),
        "size_kb": round(path.stat().st_size / 1024, 2),
    })

report = {
    "experiment": {
        "name": "Transformers ONNX CPU Export and Fusion Benchmark",
        "model": "sentence-transformers/all-MiniLM-L6-v2",
        "transformers_version": "5.23.0",
        "onnxruntime_version": "1.32.0",
        "execution_device": "CPU",
        "benchmark_configuration": {
            "warmups_per_backend": 10,
            "iterations_per_round": 100,
            "rounds": 5,
            "pytorch_intra_op_threads": 4,
            "pytorch_inter_op_threads": 1,
            "onnxruntime_intra_op_threads": 4,
            "onnxruntime_inter_op_threads": 1,
        },
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    },
    "artifacts": artifacts,
    "csv_artifacts": csv_artifacts,
    "interpretation": {
        "benchmark_scope": (
            "Transformer forward pass only. Excludes tokenization, "
            "pooling, normalization, model loading, and export time."
        ),
        "controlled_benchmark": (
            "Standard ONNX had a lower median latency than CPU-fused ONNX "
            "in this run. Timing variability was substantial."
        ),
        "graph_inspection": (
            "CPU fusion introduced BiasGelu and SkipLayerNormalization "
            "operators and reduced graph node count and file size."
        ),
        "embedding_validation": (
            "Normalized PyTorch and ONNX embeddings were numerically "
            "close, and all four test queries retained identical rankings "
            "on the small test corpus."
        ),
        "limitations": [
            "Only one model was evaluated.",
            "Only four input texts were used.",
            "Only CPU inference was evaluated.",
            "Five benchmark rounds were performed in one process, "
            "not five independent process sessions.",
            "The four-item retrieval test does not establish general "
            "retrieval quality.",
            "Results are specific to this Kaggle environment and "
            "software configuration.",
        ],
    },
}

with open(REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)

print("=" * 65)
print("FINAL EXPERIMENT REPORT")
print("=" * 65)

print("\nCollected JSON artifacts:")
for name in artifacts:
    print(" -", name)

print("\nCSV artifacts:")
for item in csv_artifacts:
    print(f" - {item['path']} ({item['size_kb']} KB)")

print("\nControlled benchmark summary:")
controlled = artifacts.get("controlled_benchmark_summary.json")
if controlled:
    print(json.dumps(controlled, indent=2))
else:
    print("Not found")

print("\nGraph comparison:")
graph = artifacts.get("graph_inspection.json")
if graph:
    for name, details in graph.items():
        print(
            f"{name}: {details.get('total_nodes')} nodes, "
            f"{details.get('file_size_mb')} MB"
        )

print("\nFinal report saved to:")
print(REPORT_PATH)


FINAL EXPERIMENT REPORT

Collected JSON artifacts:
 - baseline_metadata.json
 - benchmark_summary.json
 - controlled_benchmark_summary.json
 - graph_inspection.json
 - onnx/config.json
 - onnx/export.json
 - onnx_cpu_fused/config.json
 - onnx_cpu_fused/export.json
 - retrieval_consistency.json

CSV artifacts:
 - benchmark_comparison.csv (0.24 KB)
 - controlled_benchmark_raw.csv (39.71 KB)

Controlled benchmark summary:
{
  "benchmark_type": "controlled repeated measurement rounds",
  "independent_process_sessions": false,
  "rounds": 5,
  "warmups_per_backend": 10,
  "iterations_per_backend_per_round": 100,
  "pytorch_threads": 4,
  "onnx_intra_op_threads": 4,
  "onnx_inter_op_threads": 1,
  "onnx_execution_mode": "ORT_SEQUENTIAL",
  "hardware_logical_cores": 4,
  "onnxruntime_version": "1.32.0",
  "results": {
    "pytorch": {
      "measurements": 500,
      "overall_median_ms": 172.62393600000001,
      "mean_ms": 153.187345824,
      "p95_ms": 213.5712622,
      "iqr_ms": 50.752318

In [13]:

from pathlib import Path
import json
import pandas as pd
import torch
import platform

BASE_DIR = Path("/kaggle/working/experiment01")

print("=" * 70)
print("1. CURRENT ENVIRONMENT")
print("=" * 70)

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("PyTorch intra-op threads:", torch.get_num_threads())
print("PyTorch inter-op threads:", torch.get_num_interop_threads())

print("\n" + "=" * 70)
print("2. EARLIER BENCHMARK SUMMARY")
print("=" * 70)

old_summary_path = BASE_DIR / "benchmark_summary.json"

if old_summary_path.exists():
    with open(old_summary_path) as f:
        old_summary = json.load(f)
    print(json.dumps(old_summary, indent=2))
else:
    print("Earlier benchmark summary not found.")

print("\n" + "=" * 70)
print("3. EARLIER BENCHMARK CSV")
print("=" * 70)

old_csv = BASE_DIR / "benchmark_comparison.csv"

if old_csv.exists():
    old_df = pd.read_csv(old_csv)
    print("Columns:", list(old_df.columns))
    print(old_df.to_string(index=False))
else:
    print("Earlier benchmark CSV not found.")

print("\n" + "=" * 70)
print("4. CONTROLLED RAW MEASUREMENTS")
print("=" * 70)

raw_path = BASE_DIR / "controlled_benchmark_raw.csv"

if raw_path.exists():
    raw_df = pd.read_csv(raw_path)
    print("Columns:", list(raw_df.columns))
    print("Total measurements:", len(raw_df))

    print("\nFirst five rows:")
    print(raw_df.head().to_string(index=False))

    print("\nNumeric column summary:")
    print(raw_df.describe(include="all").to_string())

    # Identify likely backend and latency columns.
    backend_col = next(
        (c for c in raw_df.columns
         if c.lower() in ["backend", "model", "runtime"]),
        None
    )
    latency_col = next(
        (c for c in raw_df.columns
         if any(x in c.lower() for x in ["latency", "duration", "time_ms"])),
        None
    )

    if backend_col and latency_col:
        print("\nLatency by backend (ms):")
        print(
            raw_df.groupby(backend_col)[latency_col]
            .agg(["count", "median", "mean", "std", "min", "max"])
            .to_string()
        )
else:
    print("Controlled raw CSV not found.")

print("\n" + "=" * 70)
print("5. BASELINE METADATA")
print("=" * 70)

metadata_path = BASE_DIR / "baseline_metadata.json"

if metadata_path.exists():
    with open(metadata_path) as f:
        print(json.dumps(json.load(f), indent=2))
else:
    print("Baseline metadata not found.")

print("\nDIAGNOSTIC COMPLETE — no inference was executed.")


1. CURRENT ENVIRONMENT
Python: 3.13.18
PyTorch: 2.11.0+cpu
PyTorch intra-op threads: 4
PyTorch inter-op threads: 1

2. EARLIER BENCHMARK SUMMARY
{
  "iterations": 100,
  "warmup": 10,
  "pytorch": {
    "mean_ms": 23.276184179997017,
    "median_ms": 22.939216000025908,
    "p95_ms": 25.25485929998581,
    "throughput_texts_sec": 171.84947365374012
  },
  "onnx": {
    "mean_ms": 19.246437800001672,
    "median_ms": 19.013154999981907,
    "p95_ms": 20.398136750000617,
    "throughput_texts_sec": 207.83066672211172
  },
  "speedup": 1.2093762192188622,
  "max_absolute_difference": 8.791685104370117e-07,
  "mean_absolute_difference": 2.0039327353060798e-07,
  "mean_cosine_similarity": 0.9999999403953552,
  "minimum_cosine_similarity": 0.9999998807907104
}

3. EARLIER BENCHMARK CSV
Columns: ['mean_ms', 'median_ms', 'p95_ms', 'throughput_texts_sec']
  mean_ms  median_ms    p95_ms  throughput_texts_sec
23.276184  22.939216 25.254859            171.849474
19.246438  19.013155 20.398137     